# Solar Filament Segmentation 2026 - YOLO11s (detect) -> crop -> U-Net pipeline

**Competition:** Solar Filament Segmentation Challenge 2026 (Kaggle / IEEE BigData Cup 2026)
**Dataset:** MAGFiLO_1.0_Kaggle_2026 (GONG H-Alpha full-disk images, COCO-style polygon annotations)
**Primary metric:** Panoptic Quality (PQ = SQ x RQ) with penalties for fragmentation (one GT filament split into many predictions) and over-merging (many GT filaments collapsed into one prediction), plus computational efficiency. Dice is a secondary metric.

**Approach (two-stage instance segmentation):**
1. **Detect** - YOLO11s learns to draw a tight bounding box around every individual filament. Because each detection is already a separate instance, this sidesteps the fragmentation/over-merging failure modes that plague pure semantic segmentation + connected-components approaches.
2. **Crop** - each detected box (padded) is cropped out of the full-disk image.
3. **Segment** - a small U-Net (optionally with an edge-attention bottleneck, inspired by EdgeAttNet, arXiv:2509.02964) refines the crop into a precise binary mask that captures fine filament structure (spine + barbs).
4. The per-crop masks are stitched back into full-resolution, per-instance label maps and scored with Panoptic Quality (via the `panoptica` package) and Dice, mirroring the competition grader as closely as possible pending the organizers' official self-eval notebook.

> Fill in TODOs marked below once you've confirmed the exact `sample_submission.csv` schema from the competition Data tab.

In [ ]:
import os, json, math, random, time, collections
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# --- Paths (confirmed from the attached competition dataset) ---------------
DATA_ROOT = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
TRAIN_IMG_DIR = DATA_ROOT / "train" / "train_images"
TEST_IMG_DIR = DATA_ROOT / "test" / "test_images"
TRAIN_ANN_PATH = DATA_ROOT / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

WORK_DIR = Path("/kaggle/working")
YOLO_DIR = WORK_DIR / "yolo_dataset"
CKPT_DIR = WORK_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load annotations and explore the dataset

The annotation JSON is COCO-style but each `segmentation` is a **single flat polygon**
`[x0, y0, x1, y1, ...]` (not the usual COCO list-of-polygons-per-instance), and there is
an extra `spine` field (a polyline through the filament's long axis) alongside the usual
`bbox`. `category_id` encodes chirality (Left/Right/Unidentifiable[/Ambiguous]), which we
ignore for pure segmentation but keep around in case an auxiliary head is worth adding later.

In [ ]:
with open(TRAIN_ANN_PATH) as f:
    coco = json.load(f)

images_by_id = {im["id"]: im for im in coco["images"]}
anns_by_image = collections.defaultdict(list)
for ann in coco["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

cat_names = {c["id"]: c["name"] for c in coco.get("categories", [])}

n_imgs = len(images_by_id)
n_anns = len(coco["annotations"])
print(f"{n_imgs} training images, {n_anns} filament instances "
      f"({n_anns / max(n_imgs, 1):.2f} filaments/image on average)")
print("categories:", cat_names)

counts = collections.Counter(len(v) for v in anns_by_image.values())
print("filaments-per-image histogram (count -> #images):", dict(sorted(counts.items())[:10]), "...")

# Do the annotations carry a per-annotator id? MAGFiLO images can be labeled by 1-3
# independent annotators - the organizers' self-eval script expects GT filament_id
# to encode this as "{annotator}-{image_id}_{n}" (see section 8). Detect it rather
# than assume, and fall back to a constant if this release doesn't expose it.
_sample_ann = coco["annotations"][0]
print("annotation fields:", list(_sample_ann.keys()))
ANNOTATOR_KEY = next((k for k in ("annotator_id", "annotator", "worker_id", "user_id")
                      if k in _sample_ann), None)
print("detected annotator field:", ANNOTATOR_KEY, "(None -> every image treated as single-annotator)")

In [ ]:
def polygon_to_mask(segmentation, height, width):
    """segmentation: flat [x0,y0,x1,y1,...] list (MAGFiLO format) -> uint8 HxW binary mask."""
    pts = np.asarray(segmentation, dtype=np.float32).reshape(-1, 2)
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(mask, [pts.round().astype(np.int32)], 1)
    return mask


def load_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


# Visualize one densely-annotated example
example_id = max(anns_by_image, key=lambda k: len(anns_by_image[k]))
example_info = images_by_id[example_id]
img = load_image(TRAIN_IMG_DIR / example_info["file_name"])
H, W = img.shape

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(img, cmap="gray")
axes[0].set_title(f"{example_info['file_name']} ({len(anns_by_image[example_id])} filaments)")
axes[1].imshow(img, cmap="gray")
for ann in anns_by_image[example_id]:
    pts = np.asarray(ann["segmentation"], dtype=np.float32).reshape(-1, 2)
    axes[1].plot(pts[:, 0], pts[:, 1], linewidth=1, color="red")
    x, y, w, h = ann["bbox"]
    axes[1].add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="yellow", linewidth=0.8))
axes[1].set_title("polygon (red) + bbox (yellow)")
for a in axes:
    a.axis("off")
plt.tight_layout()
plt.show()

## 2. Train / validation split

Split at the **image** level (not the instance level) so no filament from a validation
image leaks into training. A random 90/10 split is a reasonable default; consider a
time-based split instead (e.g. hold out the most recent year) if you want the validation
score to better reflect generalization to new solar-cycle phases.

In [ ]:
image_ids = list(images_by_id.keys())
random.Random(SEED).shuffle(image_ids)
n_val = max(1, int(0.1 * len(image_ids)))
val_ids = set(image_ids[:n_val])
train_ids = set(image_ids[n_val:])
print(f"train images: {len(train_ids)}  val images: {len(val_ids)}")

with open(WORK_DIR / "split.json", "w") as f:
    json.dump({"train": sorted(train_ids), "val": sorted(val_ids)}, f)

## 3. Stage 1 - YOLO11s filament detector

Convert every polygon's bounding box into YOLO-format labels (single class, `filament`)
and materialize the directory layout Ultralytics expects. Images are **symlinked**
(not copied) from the read-only `/kaggle/input` tree to save disk and time.

Augmentation notes for this domain:
- `fliplr` / `flipud` are safe for detection-only boxes (we are not predicting chirality here).
- `mosaic` is disabled: filaments are large-scale, disk-relative structures and mosaic's
  4-image stitching creates physically nonsensical solar disks.
- `hsv_h` / `hsv_s` are zeroed since the source imagery is single-channel H-alpha (no
  meaningful hue/saturation) even though it's stored as a JPEG.

In [ ]:
def build_yolo_dataset():
    for split, ids in [("train", train_ids), ("val", val_ids)]:
        (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img_id in ids:
            info = images_by_id[img_id]
            src = TRAIN_IMG_DIR / info["file_name"]
            dst_img = YOLO_DIR / "images" / split / info["file_name"]
            if not dst_img.exists():
                os.symlink(src, dst_img)

            w, h = info["width"], info["height"]
            lines = []
            for ann in anns_by_image[img_id]:
                x, y, bw, bh = ann["bbox"]
                cx, cy = (x + bw / 2) / w, (y + bh / 2) / h
                nw, nh = bw / w, bh / h
                lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

            label_path = YOLO_DIR / "labels" / split / (Path(info["file_name"]).stem + ".txt")
            label_path.write_text("\n".join(lines))

    data_yaml = f"""path: {YOLO_DIR}
train: images/train
val: images/val
names:
  0: filament
"""
    (YOLO_DIR / "data.yaml").write_text(data_yaml)
    print("YOLO dataset ready at", YOLO_DIR)


build_yolo_dataset()

In [ ]:
# pip install -q ultralytics   # uncomment if not already available in the environment
from ultralytics import YOLO

yolo_model = YOLO("yolo11s.pt")  # pretrained COCO checkpoint, fine-tuned below

yolo_results = yolo_model.train(
    data=str(YOLO_DIR / "data.yaml"),
    imgsz=1024,          # full 2048x2048 images downsized; raise if VRAM allows, it helps thin filaments
    epochs=60,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",  # Kaggle: make sure Settings > Accelerator = GPU is on
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
    patience=15,
    mosaic=0.0,
    degrees=10, translate=0.05, scale=0.2, shear=0.0, perspective=0.0,
    flipud=0.5, fliplr=0.5,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.1,
    project=str(WORK_DIR / "runs"),
    name="yolo11s_filament_detect",
)

YOLO_WEIGHTS = WORK_DIR / "runs" / "yolo11s_filament_detect" / "weights" / "best.pt"

## 4. Stage 2 - crop dataset + U-Net for fine segmentation

For every ground-truth instance, crop a padded region around its bounding box (so the
detector's imprecision at inference time is covered) and rasterize the matching polygon
mask inside that same crop.

**Aspect-preserving (letterboxed) crop, not a stretched square.** Filaments are long and
thin - a naive square crop around an elongated bbox wastes most of its resolution on
empty margin along the short axis once resized to `CROP_SIZE`, which directly hurts the
model's ability to resolve barbs. Instead we pad width/height independently, then
letterbox-resize (scale to fit, pad with black) into a square `CROP_SIZE x CROP_SIZE`
canvas - this keeps the filament's true aspect ratio undistorted while using the
available pixels much more efficiently than the old center-square-then-resize approach.

In [ ]:
CROP_SIZE = 256
PAD_FACTOR = 1.35   # crop box is 35% larger than the tight bbox on each side
MIN_CROP_SIDE = 48  # floor so tiny filaments still get a usable crop region


def padded_rect_crop(x, y, w, h, W, H, pad_factor=PAD_FACTOR, min_side=MIN_CROP_SIDE):
    """Independently-padded rectangle around a bbox (NOT forced to a square) -
    preserves the filament's true aspect ratio; see letterbox_resize below."""
    cx, cy = x + w / 2, y + h / 2
    pad_w, pad_h = max(w * pad_factor, min_side), max(h * pad_factor, min_side)
    x1, y1 = max(0, int(round(cx - pad_w / 2))), max(0, int(round(cy - pad_h / 2)))
    x2, y2 = min(W, int(round(cx + pad_w / 2))), min(H, int(round(cy + pad_h / 2)))
    return x1, y1, x2, y2


def letterbox_resize(img, size=CROP_SIZE, interp=cv2.INTER_LINEAR):
    """Scale `img` so its longer side == size, pad the rest with zeros to reach a
    size x size square. Returns (canvas, scale, left, top) - the last three let you
    invert the transform exactly (see undo_letterbox)."""
    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=interp)
    canvas = np.zeros((size, size), dtype=img.dtype)
    top, left = (size - new_h) // 2, (size - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized
    return canvas, scale, left, top


def undo_letterbox(canvas, scale, left, top, orig_w, orig_h, interp=cv2.INTER_NEAREST):
    new_w, new_h = max(1, round(orig_w * scale)), max(1, round(orig_h * scale))
    cropped = canvas[top:top + new_h, left:left + new_w]
    return cv2.resize(cropped, (orig_w, orig_h), interpolation=interp)


class FilamentCropDataset(Dataset):
    def __init__(self, image_ids, augment=False):
        self.samples = []  # (image_id, ann)
        for img_id in image_ids:
            for ann in anns_by_image[img_id]:
                self.samples.append((img_id, ann))
        self.augment = augment
        self._img_cache = {}

    def __len__(self):
        return len(self.samples)

    def _get_image(self, img_id):
        if img_id not in self._img_cache:
            info = images_by_id[img_id]
            self._img_cache[img_id] = load_image(TRAIN_IMG_DIR / info["file_name"])
        return self._img_cache[img_id]

    def __getitem__(self, idx):
        img_id, ann = self.samples[idx]
        img = self._get_image(img_id)
        H, W = img.shape
        full_mask = polygon_to_mask(ann["segmentation"], H, W)

        x, y, w, h = ann["bbox"]
        x1, y1, x2, y2 = padded_rect_crop(x, y, w, h, W, H)
        crop = img[y1:y2, x1:x2]
        mask = full_mask[y1:y2, x1:x2]

        crop, _, _, _ = letterbox_resize(crop, CROP_SIZE, cv2.INTER_LINEAR)
        mask, _, _, _ = letterbox_resize(mask, CROP_SIZE, cv2.INTER_NEAREST)

        if self.augment:
            if random.random() < 0.5:
                crop, mask = crop[:, ::-1].copy(), mask[:, ::-1].copy()
            if random.random() < 0.5:
                crop, mask = crop[::-1, :].copy(), mask[::-1, :].copy()
            k = random.choice([0, 1, 2, 3])
            if k:
                crop = np.rot90(crop, k).copy()
                mask = np.rot90(mask, k).copy()
            if random.random() < 0.3:
                gain = random.uniform(0.85, 1.15)
                bias = random.uniform(-10, 10)
                crop = np.clip(crop.astype(np.float32) * gain + bias, 0, 255).astype(np.uint8)

        crop_t = torch.from_numpy(crop).float().unsqueeze(0) / 255.0
        mask_t = torch.from_numpy(mask).float().unsqueeze(0)
        return crop_t, mask_t


train_crop_ds = FilamentCropDataset(train_ids, augment=True)
val_crop_ds = FilamentCropDataset(val_ids, augment=False)
print(f"train crops: {len(train_crop_ds)}  val crops: {len(val_crop_ds)}")

train_loader = DataLoader(train_crop_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_crop_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

## 5. Model - compact U-Net with an optional edge-guided attention bottleneck

Plain U-Net decoder/encoder with a small self-attention block at the bottleneck whose
Query/Key are biased by a learned edge map of the *input* crop (idea from EdgeAttNet,
arXiv:2509.02964) - this is specifically meant to help recover thin barbs, which is
exactly the kind of fine structure the competition's Dice/PQ scoring rewards. Keep the
channel widths small (32-256) since the efficiency of the full pipeline is also scored.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class EdgeGuidedAttention(nn.Module):
    """Self-attention over the bottleneck feature map, with Q/K biased by an edge prior."""

    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.edge_conv = nn.Conv2d(1, 1, 3, padding=1)
        self.edge_proj = nn.Conv2d(1, channels, 1)
        self.mhsa = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(channels)

    def forward(self, feat, input_image):
        B, C, h, w = feat.shape
        edge = torch.sigmoid(self.edge_conv(input_image))
        edge = F.interpolate(edge, size=(h, w), mode="bilinear", align_corners=False)
        edge = self.edge_proj(edge)  # B,C,h,w

        tokens = feat.flatten(2).transpose(1, 2)          # B, h*w, C
        edge_tok = edge.flatten(2).transpose(1, 2)         # B, h*w, C
        q = k = tokens + edge_tok
        attn_out, _ = self.mhsa(q, k, tokens)
        tokens = self.norm(tokens + attn_out)
        return tokens.transpose(1, 2).reshape(B, C, h, w)


class EdgeAttUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32, use_edge_attention=True):
        super().__init__()
        self.use_edge_attention = use_edge_attention
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base * 8, base * 16)
        if use_edge_attention:
            self.edge_attn = EdgeGuidedAttention(base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = ConvBlock(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.head = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        if self.use_edge_attention:
            b = self.edge_attn(b, x)

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


unet_model = EdgeAttUNet(use_edge_attention=True).to(DEVICE)
n_params = sum(p.numel() for p in unet_model.parameters())
print(f"U-Net parameters: {n_params / 1e6:.2f}M")

## 6. Loss + training loop

`BCE + Dice` (as in EdgeAttNet) balances pixel-wise correctness with the shape-overlap
objective the competition actually scores on.

In [ ]:
def dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    num = 2 * (prob * target).sum(dim=(1, 2, 3)) + eps
    den = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + eps
    return 1 - (num / den).mean()


def bce_dice_loss(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)


@torch.no_grad()
def dice_score(logits, target, thresh=0.5, eps=1e-6):
    pred = (torch.sigmoid(logits) > thresh).float()
    num = 2 * (pred * target).sum(dim=(1, 2, 3)) + eps
    den = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + eps
    return (num / den).mean().item()


def train_unet(model, train_loader, val_loader, epochs=40, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

    best_val_dice = 0.0
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for crops, masks in train_loader:
            crops, masks = crops.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
                logits = model(crops)
                loss = bce_dice_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running_loss += loss.item() * crops.size(0)
        sched.step()

        model.eval()
        val_dice, n = 0.0, 0
        with torch.no_grad():
            for crops, masks in val_loader:
                crops, masks = crops.to(DEVICE), masks.to(DEVICE)
                logits = model(crops)
                val_dice += dice_score(logits, masks) * crops.size(0)
                n += crops.size(0)
        val_dice /= max(n, 1)

        print(f"epoch {epoch+1:03d}/{epochs}  train_loss={running_loss/len(train_loader.dataset):.4f}  val_dice={val_dice:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), CKPT_DIR / "unet_best.pt")

    print("best val dice:", best_val_dice)
    return model


unet_model = train_unet(unet_model, train_loader, val_loader, epochs=40)
unet_model.load_state_dict(torch.load(CKPT_DIR / "unet_best.pt"))
unet_model.eval()

## 7. Full inference pipeline (detect -> crop -> segment -> stitch)

Detections are processed **highest-confidence first**; when two predicted instances
overlap, the higher-confidence one wins the contested pixels. This keeps the output
non-overlapping (simplifying the instance label map fed to PQ) without needing a
separate NMS-on-masks step.

In [ ]:
MIN_INSTANCE_AREA = 25  # drop spurious specks smaller than this many pixels


@torch.no_grad()
def predict_instances(image_path, yolo_model, unet_model, conf_thres=0.25, iou_thres=0.5, imgsz=1024,
                       crop_size=CROP_SIZE, pad_factor=PAD_FACTOR, mask_thresh=0.5,
                       min_instance_area=MIN_INSTANCE_AREA):
    img = load_image(image_path)
    H, W = img.shape

    det = yolo_model.predict(img, conf=conf_thres, iou=iou_thres, imgsz=imgsz, verbose=False)[0]
    if det.boxes is None or len(det.boxes) == 0:
        return np.zeros((H, W), dtype=np.uint16), []

    boxes = det.boxes.xyxy.cpu().numpy()
    scores = det.boxes.conf.cpu().numpy()
    order = np.argsort(-scores)

    label_map = np.zeros((H, W), dtype=np.uint16)
    instance_scores = []
    next_id = 1

    for idx in order:
        x1b, y1b, x2b, y2b = boxes[idx]
        x, y, w, h = x1b, y1b, x2b - x1b, y2b - y1b
        cx1, cy1, cx2, cy2 = padded_rect_crop(x, y, w, h, W, H, pad_factor)
        if cx2 <= cx1 or cy2 <= cy1:
            continue

        crop = img[cy1:cy2, cx1:cx2]
        crop_letterboxed, scale, left, top = letterbox_resize(crop, crop_size, cv2.INTER_LINEAR)
        crop_t = torch.from_numpy(crop_letterboxed).float().unsqueeze(0).unsqueeze(0).to(DEVICE) / 255.0

        logits = unet_model(crop_t)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
        mask_letterboxed = (prob > mask_thresh).astype(np.uint8)
        mask_full_crop = undo_letterbox(mask_letterboxed, scale, left, top, cx2 - cx1, cy2 - cy1, cv2.INTER_NEAREST)

        if mask_full_crop.sum() < min_instance_area:
            continue

        region = label_map[cy1:cy2, cx1:cx2]
        free = (mask_full_crop > 0) & (region == 0)
        if free.sum() < min_instance_area:
            continue
        region[free] = next_id
        label_map[cy1:cy2, cx1:cx2] = region
        instance_scores.append(scores[idx])
        next_id += 1

    return label_map, instance_scores

## 8. Self-evaluation - using the organizers' own self-evaluation notebook

The organizers published an official self-evaluation notebook
(https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook) that takes a
`pred_df` and `gt_df` - **exactly the submission-format** `filament_id`/`segmentation_rle`
dataframes - and reports PQ, the IoU/Dice distributions, and the one-to-many /
many-to-one match counts. Rather than approximate that logic, the helper functions below
are copied verbatim from it, so our local score is computed exactly like the grader:

- PQ is a **single global score**: TP/FP/FN are pooled across every image first, then
  `PQ = sum(IoU of TP pairs) / (|TP| + 0.5|FP| + 0.5|FN|)` - this is *not* the mean of
  per-image PQ values (that would be a different, more lenient number).
- Matching is a **plain IoU > 0.5 threshold**, not a 1-to-1 Hungarian/greedy match - a
  prediction that overlaps two different GT filaments above 0.5 IoU each (an over-merge)
  contributes to more than one "hit", which is intentional: it's exactly the many-to-one
  case the `plot_m2n_counts` chart is built to surface.
- Ground truth rows are keyed by `"{annotator}-{image_id}_{n}"` because MAGFiLO images
  can carry 1-3 independent annotator labelings; predictions are only keyed by
  `image_id` and get compared against *each* annotator's set separately. Our train JSON
  is checked above for an explicit annotator field - if it doesn't have one, every GT
  instance is treated as coming from a single "1-" annotator, which is the correct
  simplification for a single-annotation-per-image release.

In [ ]:
# --- Verbatim from the organizers' self-evaluation notebook -----------------
# https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook
# (kept unmodified so our local PQ/plots match the real grader exactly)

from pycocotools import mask as mask_util
from collections import Counter
from matplotlib.patches import Patch


def get_overlap_matrices(gt_layers: torch.Tensor, pred_layers: torch.Tensor):
    """Pairwise IoU and Dice between GT (n_gt,H,W) and predicted (n_pred,H,W) mask layers."""
    n_gt, height, width = gt_layers.shape
    n_pred = pred_layers.shape[0]

    gt_flat = gt_layers.reshape(n_gt, height * width)
    pred_flat = pred_layers.reshape(n_pred, height * width)

    intersection = torch.matmul(gt_flat, pred_flat.t())

    gt_areas = gt_flat.sum(dim=1).view(-1, 1)
    pred_areas = pred_flat.sum(dim=1).view(1, -1)

    union = gt_areas + pred_areas - intersection

    iou_matrix = torch.where(union == 0, torch.tensor(0., device=gt_layers.device), intersection / union)
    dice_matrix = torch.where(union == 0, torch.tensor(0., device=gt_layers.device),
                               2 * intersection / (gt_areas + pred_areas))
    return iou_matrix, dice_matrix


def fp_count_hit(hit_matrix: torch.Tensor) -> int:
    """Predictions (columns) with no qualifying GT overlap."""
    return (hit_matrix.sum(dim=0) == 0).sum().item()


def fn_count_hit(hit_matrix: torch.Tensor) -> int:
    """GT filaments (rows) with no qualifying predicted overlap."""
    return (hit_matrix.sum(dim=1) == 0).sum().item()


def rles_to_layers(rles: list, height: int = 2048, width: int = 2048) -> np.ndarray:
    """Decode compressed COCO RLE strings into an (n_masks, H, W) binary stack."""
    if not rles:
        return np.zeros((0, height, width), dtype=np.float32)
    rle_dicts = [{"size": [height, width], "counts": rle} for rle in rles]
    masks = mask_util.decode(rle_dicts)  # (H, W, n_masks)
    return masks.transpose(2, 0, 1).astype(np.float32)


def process_entry(annotator_image, gt_annotator_image_ids, pred_image_ids, gt_df, pred_df):
    image_id = annotator_image.split("-", maxsplit=1)[1]

    gt_rles = gt_df.loc[gt_annotator_image_ids == annotator_image, "segmentation_rle"].tolist()
    gt_layers = torch.from_numpy(rles_to_layers(gt_rles))

    pred_rles = pred_df.loc[pred_image_ids == image_id, "segmentation_rle"].tolist()
    pred_layers = torch.from_numpy(rles_to_layers(pred_rles))

    n_gt, n_pred = len(gt_rles), len(pred_rles)
    iou_matrix, dice_matrix = get_overlap_matrices(gt_layers, pred_layers)
    return iou_matrix, dice_matrix, n_gt, n_pred


def get_overlap_df(gt_df: pd.DataFrame, pred_df: pd.DataFrame) -> pd.DataFrame:
    gt_annotator_image_ids = gt_df["filament_id"].str.split("_", n=1).str[0]
    pred_image_ids = pred_df["filament_id"].str.split("_", n=1).str[0]

    annotator_images = gt_annotator_image_ids.unique()
    overlap_df = pd.DataFrame({"annotator_image": annotator_images})

    results = overlap_df["annotator_image"].apply(
        process_entry,
        gt_annotator_image_ids=gt_annotator_image_ids,
        pred_image_ids=pred_image_ids,
        gt_df=gt_df,
        pred_df=pred_df,
    )
    (overlap_df["iou_matrix"], overlap_df["dice_matrix"],
     overlap_df["n_gt"], overlap_df["n_pred"]) = zip(*results)
    return overlap_df


def get_pq_score(overlap_df: pd.DataFrame) -> float:
    iou_threshold = 0.5
    tp_iou_scores, fp_count, fn_count = [], 0, 0

    for row in overlap_df.itertuples(index=False):
        iou_matrix, n_gt, n_pred = row.iou_matrix, row.n_gt, row.n_pred
        if n_gt == 0:
            fp_count += n_pred
            continue
        if n_pred == 0:
            fn_count += n_gt
            continue
        hit_matrix = iou_matrix > iou_threshold
        tp_iou_scores.extend(iou_matrix[hit_matrix].tolist())
        fp_count += fp_count_hit(hit_matrix)
        fn_count += fn_count_hit(hit_matrix)

    tp_count = len(tp_iou_scores)
    denominator = tp_count + 0.5 * fp_count + 0.5 * fn_count
    return (sum(tp_iou_scores) / denominator) if denominator > 0 else 0.0


def plot_distribution(matrices: pd.Series, title: str, x_label: str):
    scores = np.concatenate([np.asarray(m, dtype=float).ravel() for m in matrices] or [np.zeros(0)])
    scores = scores[scores > 0]
    x_max = 1.0 if scores.size == 0 or scores.max() <= 1.0 else scores.max()
    if x_max <= 1.0:
        bins = np.linspace(0.0, x_max, 51)
        bar_width = 0.02
    else:
        bins = np.arange(0, int(x_max) + 2)
        bar_width = 0.8
    counts, edges = np.histogram(scores, bins=bins)
    probabilities = counts / counts.sum() if counts.sum() > 0 else counts
    centers = (edges[:-1] + edges[1:]) / 2.0
    plt.bar(centers, probabilities, width=bar_width, edgecolor="black", alpha=0.7, color="navy")
    plt.grid(axis="y", linestyle="--")
    if scores.size > 0:
        mean_score = scores.mean()
        plt.axvline(mean_score, color="crimson", linestyle="--", linewidth=1.5, label=f"Mean = {mean_score:.3f}")
        plt.legend(fontsize=8)
    plt.xlabel(x_label); plt.ylabel("Probability"); plt.title(title)


def plot_m2n_counts(overlap_df: pd.DataFrame, title: str = "Many to One and One to Many Distribution"):
    offset = 0.5
    gt_match_counts, pred_match_counts = [], []
    for row in overlap_df.itertuples(index=False):
        iou_matrix, n_gt, n_pred = row.iou_matrix, row.n_gt, row.n_pred
        if n_gt == 0:
            pred_match_counts.extend([0] * n_pred)
            continue
        if n_pred == 0:
            gt_match_counts.extend([0] * n_gt)
            continue
        hit_matrix = iou_matrix > 0
        gt_match_counts.extend(hit_matrix.sum(dim=1).tolist())
        pred_match_counts.extend(hit_matrix.sum(dim=0).tolist())

    gt_degree_counts = Counter(gt_match_counts)
    pred_degree_counts = Counter(pred_match_counts)
    gt_miss_count = gt_degree_counts.get(0, 0)
    gt_hit_count = sum(c for d, c in gt_degree_counts.items() if d > 0)
    pred_miss_count = pred_degree_counts.get(0, 0)
    pred_hit_count = sum(c for d, c in pred_degree_counts.items() if d > 0)

    gt_bars = {d + offset: c for d, c in gt_degree_counts.items()}
    pred_bars = {-(d + offset): c for d, c in pred_degree_counts.items()}
    bar_width = 0.8

    for pos in sorted(gt_bars):
        degree = int(round(pos - offset))
        plt.bar(pos, gt_bars[pos], width=bar_width,
                color="crimson" if degree == 0 else "forestgreen", edgecolor="black", alpha=0.7)
    for pos in sorted(pred_bars):
        degree = int(round(-pos - offset))
        plt.bar(pos, pred_bars[pos], width=bar_width,
                color="darkorange" if degree == 0 else "steelblue", edgecolor="black", alpha=0.7)

    plt.axvline(0, color="black", linewidth=0.8)
    plt.grid(axis="y", linestyle="--")
    plt.xlabel("Matching Case"); plt.ylabel("Frequency")
    positions = sorted(gt_bars) + sorted(pred_bars)
    labels = [f"{abs(int(round(v + offset)))}:1" if v < 0 else f"1:{int(round(v - offset))}" for v in positions]
    plt.xticks(positions, labels, rotation=45, fontsize=9)
    plt.title(title)
    max_pos = max([abs(k) for k in positions], default=offset)
    plt.xlim(-max_pos - bar_width, max_pos + bar_width)
    plt.legend(handles=[
        Patch(facecolor="forestgreen", edgecolor="black", alpha=0.7, label=f"GT Hits (1:n, n>0): {gt_hit_count}"),
        Patch(facecolor="crimson", edgecolor="black", alpha=0.7, label=f"GT Misses (1:0): {gt_miss_count}"),
        Patch(facecolor="darkorange", edgecolor="black", alpha=0.7, label=f"Prediction Misses (0:1): {pred_miss_count}"),
        Patch(facecolor="steelblue", edgecolor="black", alpha=0.7, label=f"Prediction Hits (n:1, n>0): {pred_hit_count}"),
    ], fontsize=9)


def encode_rle_counts(binary_mask):
    """pycocotools compressed-RLE 'counts' string for a HxW {0,1} mask (reused in section 9 too)."""
    rle = mask_util.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    counts = rle["counts"]
    return counts.decode("utf-8") if isinstance(counts, bytes) else counts

### Quick check first

Running the full validation set through detect -> crop -> segment can take a while.
Run this on a handful of images first to make sure the whole scoring chain works and
to get a ballpark PQ in seconds - then run the full pass below when you're ready for
the real number.

In [ ]:
QUICK_CHECK_N = 8
quick_ids = random.sample(sorted(val_ids), k=min(QUICK_CHECK_N, len(val_ids)))

gt_df_quick, pred_df_quick = build_val_gt_pred_dataframes(quick_ids)
print(f"quick check on {len(quick_ids)} image(s): "
      f"{len(gt_df_quick)} GT filaments, {len(pred_df_quick)} predicted filaments")

overlap_df_quick = get_overlap_df(gt_df_quick, pred_df_quick)
print(f"quick PQ score: {get_pq_score(overlap_df_quick):.4f}  (small sample - noisy, just a sanity check)")

In [ ]:
def build_val_gt_pred_dataframes(image_ids):
    """Ground-truth and prediction dataframes in the exact submission schema
    (filament_id, segmentation_rle), so we can feed them straight into the
    organizers' own scoring functions above."""
    gt_rows, pred_rows = [], []
    for img_id in image_ids:
        info = images_by_id[img_id]
        H, W = info["height"], info["width"]
        image_stem = Path(info["file_name"]).stem

        for n, ann in enumerate(anns_by_image[img_id], start=1):
            annotator = ann.get(ANNOTATOR_KEY, 1) if ANNOTATOR_KEY else 1
            mask = polygon_to_mask(ann["segmentation"], H, W)
            gt_rows.append({"filament_id": f"{annotator}-{image_stem}_{n}",
                             "segmentation_rle": encode_rle_counts(mask)})

        label_map, _ = predict_instances(TRAIN_IMG_DIR / info["file_name"], yolo_model, unet_model)
        for inst_id in range(1, int(label_map.max()) + 1):
            mask = label_map == inst_id
            if not mask.any():
                continue
            pred_rows.append({"filament_id": f"{image_stem}_{inst_id}",
                               "segmentation_rle": encode_rle_counts(mask)})

    cols = ["filament_id", "segmentation_rle"]
    return pd.DataFrame(gt_rows, columns=cols), pd.DataFrame(pred_rows, columns=cols)


gt_df, pred_df = build_val_gt_pred_dataframes(val_ids)
print(f"val GT filaments: {len(gt_df)}   val predicted filaments: {len(pred_df)}")

overlap_df = get_overlap_df(gt_df, pred_df)

pq_score = get_pq_score(overlap_df)
print(f"PQ Score (val): {pq_score:.4f}")

plt.figure(figsize=(6, 4)); plot_distribution(overlap_df["iou_matrix"], "IoU Distribution (val)", "IoU"); plt.show()
plt.figure(figsize=(6, 4)); plot_distribution(overlap_df["dice_matrix"], "Dice Distribution (val)", "Dice"); plt.show()
plt.figure(figsize=(8, 5)); plot_m2n_counts(overlap_df, "Many-to-One / One-to-Many Distribution (val)"); plt.show()

## 9. Test-set predictions and submission

Confirmed submission format (from the competition rules): a single CSV with two columns,
`filament_id` (`{image_id}_{n}`, 1-indexed per image, uniqueness is all that matters) and
`segmentation_rle` (pycocotools **compressed RLE counts only** - no `size`, since it's fixed
at 2048x2048 for every image).

In [ ]:
# `encode_rle_counts` and `mask_util` were already defined in section 8.

test_image_paths = sorted(TEST_IMG_DIR.glob("*.jpeg"))
submission_rows = []

for path in test_image_paths:
    image_id = path.stem  # e.g. "20150125172714Mh"
    label_map, _ = predict_instances(path, yolo_model, unet_model)
    for inst_id in range(1, int(label_map.max()) + 1):
        mask = label_map == inst_id
        if not mask.any():
            continue
        submission_rows.append({
            "filament_id": f"{image_id}_{inst_id}",
            "segmentation_rle": encode_rle_counts(mask),
        })

submission_df = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
submission_df.to_csv(WORK_DIR / "submission.csv", index=False)
submission_df.head()

## Closing the gap to the leaderboard - a prioritized punch list

(Written after v0.01 scored 0.32 on Kaggle vs. a leader at 0.55.) A few notes on
diagnosing *where* points are going before changing more:

1. **Check which metric 0.32 actually is.** Per the rules, the public scoreboard shows
   mean **Dice** on ~50% of test images - final ranking is **PQ**, which is usually
   noticeably lower than Dice for the same predictions because PQ additionally punishes
   fragmentation/over-merging (see the math in section 8: a filament split into two
   boxes typically becomes 1 FN + 2 FP, not partial credit). Run the full section-8
   self-eval on your validation split and compare its PQ *and* Dice numbers to 0.32 -
   that tells you whether the bottleneck is segmentation quality (Dice low too) or
   instance-level fragmentation/merging (Dice okay, PQ much worse).
2. **Detection recall is the ceiling for everything downstream.** If YOLO misses a
   filament, no U-Net can recover it (it's a guaranteed FN). Check the YOLO training
   run's own precision/recall/mAP - if recall is weak, try a lower `conf_thres` at
   inference (this notebook defaults to 0.25; try 0.15-0.2 and see if PQ improves or
   the extra false positives cancel it out), train at a higher `imgsz` (1024 -> 1280),
   or try `yolo11m.pt` for more capacity if the efficiency budget allows.
3. **Fragmentation/over-merging fixes**, since PQ punishes these hardest: tune
   `iou_thres` in `predict_instances` (YOLO's own box-NMS) - too low merges nearby
   filaments' boxes into one, too high leaves duplicate overlapping boxes for one long
   filament; `MIN_INSTANCE_AREA` now filters out tiny spurious predicted specks that
   would otherwise cost a free 0.5 FP penalty each.
4. **The crop is now aspect-preserving (letterboxed)** instead of a square stretch -
   this should measurably help barb/fine-structure Dice on elongated filaments without
   any retraining changes beyond re-running with the new `FilamentCropDataset`.
5. **Spine-guided oriented crops (bigger change, likely biggest remaining lever)**:
   instead of an axis-aligned box, rotate the crop to align with the filament's spine
   direction (available in the GT annotations) before letterboxing - this gives the
   U-Net a canonical, maximally-zoomed view of every filament regardless of its
   orientation on the disk. At inference you'd estimate the angle from the YOLO box's
   own thresholded content (e.g. PCA on a first-pass mask) since you don't have GT
   spine for test images.
6. **Train longer / verify training actually completed** - if the v0.01 submission's
   run was interrupted by a Kaggle session limit or under-epoched due to earlier GPU
   issues, simply finishing 60 YOLO epochs + 40 U-Net epochs (with early stopping via
   `patience`) may close a lot of the gap on its own before touching architecture.
7. **Test-time augmentation** for the U-Net stage (average predictions over horizontal
   flip) is a cheap way to squeeze out a bit more Dice with no retraining.
8. A spine-distance-weighted loss term, or the full EdgeAttNet training recipe, are
   still open ideas if segmentation quality (not detection/fragmentation) turns out to
   be the bottleneck once you've checked #1.